# 🏗️ Notebook 1: Shopping Cart — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/shopping-cart
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

An e-commerce cart + checkout. Users add items, cart persists across sessions, inventory is reserved at checkout, payment runs, order is created.

Hardest problem: **inventory**. We can't oversell when 10k people hit *Buy now* on a flash-sale item.

## Requirements

### Functional
- Add/remove/update cart items.
- Cart survives logout/login.
- Checkout reserves stock, charges payment, creates order.
- Handle payment failures (release reservation).

### Non-functional
- No **overselling** — strict for inventory.
- Checkout is idempotent — retries must not charge twice.

## Back-of-envelope

- 100M carts, avg 4 items each → 400M rows.
- Peak checkouts: 5k/s.
- Cart read QPS: 50k/s → cache in Redis, persist to DB async.

## High-level architecture

```
  [Client]
     │
     ▼
  ┌──────────────┐
  │ API Gateway  │
  └──────┬───────┘
   ┌─────┼─────────┬──────────────┬────────────┐
   ▼     ▼         ▼              ▼            ▼
 Cart   Catalog  Inventory      Order       Payment
 Svc    Svc      Svc            Svc         Svc
   │     │         │              │            │
   ▼     ▼         ▼              ▼            ▼
 Redis  PG      Redis+PG        PG           Stripe
```

- **Cart Service** keeps cart in Redis (fast) with a PG shadow copy.
- **Inventory Service** does reservations (decrement atomically, TTL if unpaid).
- **Checkout** is a small saga: reserve → charge → confirm → (compensate on failure).

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.